# Rule-based Agent — Caro 11×11

**Tác giả:** Lê Tấn Phát  
**Phạm vi:** Giai đoạn 1 — Luật thắng ngay và chặn thua (Phần 1–4 hoàn chỉnh; Phần 5–10 sẽ hoàn thiện ở Giai đoạn 2 & 3)  
**Nguồn module:** `agents/rule_based_agent.py`  
**Lưu ý:** Mọi lần chạy nên bắt đầu bằng **Restart Kernel & Run All** để đảm bảo tính tái lập.


## Phần 1 — Câu hỏi nghiên cứu và phạm vi

**Câu hỏi trung tâm:**  
Việc bổ sung nhận diện pattern nâng cao (open-four, open-three, chuỗi có khoảng trống) có thực sự nâng cao chất lượng nước đi so với việc chỉ dùng luật thắng ngay/chặn thua kết hợp điểm vị trí cơ bản hay không?

**Phạm vi Phần 1–4 (Giai đoạn 1):**
- Xây dựng baseline chiến thuật: thắng ngay → chặn thua → chọn vị trí tốt.
- Kiểm chứng thực nghiệm cho thấy luật thắng/chặn hoạt động đúng ở cả 4 hướng và 2 phía.
- Chưa cài đặt pattern nâng cao (`use_patterns=False` ở Giai đoạn 1).

**Hạn chế đã biết ở Giai đoạn 1:**
- Nước dự phòng (fallback) chỉ dựa vào điểm vị trí đơn giản, chưa nhận diện open-four hay open-three.
- Không xử lý đe dọa kép (double threat) của đối thủ — đây là giới hạn tự nhiên của tầng luật hiện tại.
- Tập ứng viên fallback bán kính 2 có thể bỏ sót các vị trí chiến lược xa quân cờ hiện có.


## Phần 2 — Môi trường và cấu hình

In [ ]:
from pathlib import Path
import os

# Chuyển về thư mục caro-11x11 để import module chung từ vị trí notebook.
if Path.cwd().name == 'notebooks' and (Path.cwd().parent / 'core').is_dir():
    os.chdir(Path.cwd().parent)
if not (Path.cwd() / 'core' / 'game_engine.py').is_file():
    raise RuntimeError('Hãy mở notebook từ thư mục caro-11x11 hoặc notebooks')

print(f'Thư mục làm việc: {Path.cwd()}')

In [ ]:
import numpy as np

from core import CaroGame
from core.game_utils import candidate_actions
from agents import RandomAgent

# Import trực tiếp từ module rule_based_agent (không chép lại thuật toán)
from agents.rule_based_agent import (
    RuleBasedAgent,
    is_winning_move,
    find_winning_actions,
    positional_score,
    DIRECTIONS,
)
from config.game_config import BOARD_SIZE, WIN_LENGTH, PLAYER_X, PLAYER_O, EMPTY

print('Import thành công.')

In [ ]:
# ---- Cấu hình thí nghiệm ----
CANDIDATE_RADIUS = 2       # Bán kính tập ứng viên cho fallback
TIEBREAK_SEED = None       # None = tất định; đặt số nguyên để ngẫu nhiên có kiểm soát
GAME_SEED = 42             # Seed cho RandomAgent trong thi đấu thử nghiệm
WIN_LENGTH_CFG = WIN_LENGTH  # 5 quân liên tiếp

# Khởi tạo agent baseline (Giai đoạn 1: use_patterns=False)
agent = RuleBasedAgent(
    name='RuleBased-Baseline',
    candidate_radius=CANDIDATE_RADIUS,
    use_patterns=False,
    seed=TIEBREAK_SEED,
)

print(f'Tác tử: {agent.name}')
print(f'Bán kính ứng viên: {agent.candidate_radius}')
print(f'use_patterns: {agent.use_patterns}')
print(f'Seed tie-breaking: {agent.seed}')
print(f'Kích thước bàn cờ: {BOARD_SIZE}×{BOARD_SIZE}')
print(f'Độ dài thắng: {WIN_LENGTH_CFG} quân liên tiếp')

## Phần 3 — Bộ vị trí chiến thuật

Dưới đây là các tình huống thử nghiệm được dựng trực tiếp bằng numpy (không dùng CaroGame.step() để tránh ràng buộc thứ tự đặt quân). Mỗi tình huống nêu rõ:
- Trạng thái bàn cờ
- Bên đến lượt
- Danh sách nước đi chấp nhận được và lý do

In [ ]:
def make_board(x_stones, o_stones, size=11):
    """Tạo bàn cờ numpy từ danh sách tọa độ."""
    board = np.zeros((size, size), dtype=np.int8)
    for r, c in x_stones:
        board[r, c] = PLAYER_X
    for r, c in o_stones:
        board[r, c] = PLAYER_O
    return board

def show_board(board, title=''):
    """Hiển thị bàn cờ dạng text."""
    symbols = {0: '.', 1: 'X', -1: 'O'}
    if title:
        print(f'=== {title} ===')
    header = '   ' + ' '.join(f'{c:2}' for c in range(board.shape[1]))
    print(header)
    for r, row in enumerate(board):
        line = f'{r:2} ' + ' '.join(f'{symbols[int(v)]:2}' for v in row)
        print(line)
    print()

# Tình huống 1: X sắp thắng ngang
board_win_h = make_board(
    x_stones=[(5, 1), (5, 2), (5, 3), (5, 4)],
    o_stones=[(0, 0), (0, 2), (0, 4)],
)
show_board(board_win_h, 'Tình huống 1: X có 4 quân ngang hàng 5')
print(f'Nước đi chấp nhận được (X): {find_winning_actions(board_win_h, PLAYER_X)}')
print('Lý do: Bất kỳ ô nào trong tập trên đều tạo chuỗi 5 quân → X thắng ngay.')
print()

In [ ]:
# Tình huống 2: O đang đe dọa, X phải chặn
board_block = make_board(
    x_stones=[(5, 5), (6, 6), (7, 7)],
    o_stones=[(0, 7), (1, 7), (2, 7), (3, 7)],
)
show_board(board_block, 'Tình huống 2: O có 4 quân dọc cột 7, X phải chặn')
o_threats = find_winning_actions(board_block, PLAYER_O)
print(f'Đe dọa của O (nước thắng ngay cho O): {o_threats}')
print(f'X không có nước thắng: {find_winning_actions(board_block, PLAYER_X)}')
print('Lý do: X phải chặn tại một trong các ô trên để O không thắng ngay.')
print()

In [ ]:
# Tình huống 3: Xung đột ưu tiên — X thắng ngay VÀ O đang đe dọa
board_conflict = make_board(
    x_stones=[(5, 1), (5, 2), (5, 3), (5, 4)],
    o_stones=[(0, 9), (1, 9), (2, 9), (3, 9)],
)
show_board(board_conflict, 'Tình huống 3: X sắp thắng VÀ O đang đe dọa (xung đột ưu tiên)')
print(f'Nước thắng ngay của X: {find_winning_actions(board_conflict, PLAYER_X)}')
print(f'Đe dọa của O: {find_winning_actions(board_conflict, PLAYER_O)}')
print('Nước chấp nhận được: X phải chọn nước THẮNG NGAY (không được chặn O trước).')
print()

In [ ]:
# Tình huống 4: Bàn cờ trống — fallback vị trí
board_empty = np.zeros((11, 11), dtype=np.int8)
show_board(board_empty, 'Tình huống 4: Bàn cờ trống — không có đe dọa trực tiếp')
print('Nước đi chấp nhận được: (5,5) — tâm bàn cờ (điểm vị trí cao nhất).')
print('Lý do: Fallback positional_score ưu tiên tâm bàn cờ theo khoảng cách Chebyshev.')
print()

## Phần 4 — Kiểm chứng luật thắng và chặn

Trong phần này, ta quan sát trực tiếp phản ứng của `RuleBasedAgent` trên các bộ vị trí chiến thuật ở:
- Cả 4 hướng: ngang, dọc, chéo chính, chéo phụ
- Cả 2 phía: X và O
- Vị trí sát biên bàn cờ
- Chuỗi dài hơn 5 quân
- Đồng thời có cơ hội thắng lẫn đe dọa thua

In [ ]:
def test_scenario(board, player, expected_rule, acceptable_actions=None, title=''):
    """Chạy agent trên một tình huống và in kết quả quan sát."""
    _agent = RuleBasedAgent(seed=None)  # tất định
    board_copy = board.copy()
    action = _agent.choose_action(board_copy, player)
    info = _agent.explain_last_action()
    
    # Kiểm tra bất biến board
    board_unchanged = np.array_equal(board, board_copy)
    rule_ok = info['rule_triggered'] == expected_rule
    action_ok = (acceptable_actions is None) or (action in acceptable_actions)
    
    status = '✓ PASS' if (rule_ok and action_ok and board_unchanged) else '✗ FAIL'
    player_sym = 'X' if player == PLAYER_X else 'O'
    
    print(f'{status} [{title}]')
    print(f'  Bên đến lượt: {player_sym}')
    print(f'  Luật kích hoạt: {info["rule_triggered"]} (kỳ vọng: {expected_rule})')
    print(f'  Nước chọn: {action}')
    print(f'  Lý do: {info["reason"]}')
    print(f'  Board bất biến: {board_unchanged}')
    if acceptable_actions:
        print(f'  Nước đi chấp nhận được: {acceptable_actions}')
        print(f'  Nước chọn hợp lệ: {action_ok}')
    print()
    return status

print('=== PHẦN 4A: Kiểm chứng thắng ngay (win_now) — 4 hướng × 2 phía ===')
print()

# Ngang — X thắng
b = make_board([(5, 1), (5, 2), (5, 3), (5, 4)], [(0, 0), (0, 2), (0, 4)])
test_scenario(b, PLAYER_X, 'win_now', find_winning_actions(b, PLAYER_X),
              'Ngang — X thắng')

# Ngang — O thắng
b = make_board([(0, 0), (1, 1), (2, 2)], [(7, 1), (7, 2), (7, 3), (7, 4)])
test_scenario(b, PLAYER_O, 'win_now', find_winning_actions(b, PLAYER_O),
              'Ngang — O thắng')

# Dọc — X thắng
b = make_board([(0, 5), (1, 5), (2, 5), (3, 5)], [(0, 0), (0, 2), (0, 4)])
test_scenario(b, PLAYER_X, 'win_now', find_winning_actions(b, PLAYER_X),
              'Dọc — X thắng')

# Dọc — O thắng
b = make_board([(0, 0), (1, 1), (2, 2)], [(1, 3), (2, 3), (3, 3), (4, 3)])
test_scenario(b, PLAYER_O, 'win_now', find_winning_actions(b, PLAYER_O),
              'Dọc — O thắng')

# Chéo chính — X thắng
b = make_board([(0, 0), (1, 1), (2, 2), (3, 3)], [(0, 1), (1, 0), (2, 1)])
test_scenario(b, PLAYER_X, 'win_now', find_winning_actions(b, PLAYER_X),
              'Chéo chính — X thắng')

# Chéo phụ — O thắng
b = make_board([(0, 0), (1, 1), (2, 2)], [(1, 9), (2, 8), (3, 7), (4, 6)])
test_scenario(b, PLAYER_O, 'win_now', find_winning_actions(b, PLAYER_O),
              'Chéo phụ — O thắng')

In [ ]:
print('=== PHẦN 4B: Kiểm chứng chặn thua (block_urgent) — 4 hướng + biên ===')
print()

# Ngang — X chặn O
b = make_board([(0, 0), (2, 2), (4, 4)], [(7, 3), (7, 4), (7, 5), (7, 6)])
test_scenario(b, PLAYER_X, 'block_urgent', find_winning_actions(b, PLAYER_O),
              'Ngang — X chặn O')

# Dọc — X chặn O
b = make_board([(5, 0), (6, 1), (7, 2)], [(0, 7), (1, 7), (2, 7), (3, 7)])
test_scenario(b, PLAYER_X, 'block_urgent', find_winning_actions(b, PLAYER_O),
              'Dọc — X chặn O')

# Chéo chính — X chặn O
b = make_board([(0, 2), (2, 0), (8, 8)], [(1, 1), (2, 2), (3, 3), (4, 4)])
test_scenario(b, PLAYER_X, 'block_urgent', find_winning_actions(b, PLAYER_O),
              'Chéo chính — X chặn O')

# Chéo phụ — X chặn O
b = make_board([(0, 0), (1, 1), (8, 8)], [(1, 9), (2, 8), (3, 7), (4, 6)])
test_scenario(b, PLAYER_X, 'block_urgent', find_winning_actions(b, PLAYER_O),
              'Chéo phụ — X chặn O')

# Biên dưới — O sát hàng 10
b = make_board([(5, 0), (6, 1), (7, 2)], [(10, 0), (10, 1), (10, 2), (10, 3)])
test_scenario(b, PLAYER_X, 'block_urgent', find_winning_actions(b, PLAYER_O),
              'Sát biên dưới — X chặn O')

# Biên trái — O sát cột 0
b = make_board([(5, 5), (6, 6), (7, 7)], [(0, 0), (1, 0), (2, 0), (3, 0)])
test_scenario(b, PLAYER_X, 'block_urgent', find_winning_actions(b, PLAYER_O),
              'Sát biên trái — X chặn O')

In [ ]:
print('=== PHẦN 4C: Chuỗi đặc biệt — >5 quân và bị chặn 2 đầu ===')
print()

# Chuỗi >5 quân: X tại (0,1)(0,2)(0,3)(0,5)(0,6), ô (0,4) trống → nối 6
b = make_board([(0, 1), (0, 2), (0, 3), (0, 5), (0, 6)], [(2, 0), (3, 0), (4, 0)])
print(f'is_winning_move(board, 0, 4, X) = {is_winning_move(b, 0, 4, PLAYER_X)}')
test_scenario(b, PLAYER_X, 'win_now', [(0, 4)], 'Chuỗi 6 quân — X nối ở giữa')

# Chuỗi bị chặn 2 đầu: O X X . X X O, đặt X vào giữa → 5 quân liên tiếp
b = make_board([(0, 2), (0, 3), (0, 5), (0, 6)], [(0, 1), (0, 7), (2, 0)])
print(f'is_winning_move(board, 0, 4, X) = {is_winning_move(b, 0, 4, PLAYER_X)}')
test_scenario(b, PLAYER_X, 'win_now', [(0, 4)], 'Chuỗi 5 bị chặn 2 đầu — X điền vào giữa')

In [ ]:
print('=== PHẦN 4D: Xung đột ưu tiên — thắng ngay > chặn thua ===')
print()

# X có 4 quân ngang VÀ O có 4 quân dọc → X phải chọn thắng ngay
b = make_board(
    [(5, 1), (5, 2), (5, 3), (5, 4)],
    [(0, 9), (1, 9), (2, 9), (3, 9)],
)
x_wins = find_winning_actions(b, PLAYER_X)
o_threats = find_winning_actions(b, PLAYER_O)
print(f'Nước thắng ngay của X: {x_wins}')
print(f'Đe dọa của O: {o_threats}')
test_scenario(b, PLAYER_X, 'win_now', x_wins, 'Xung đột: X thắng > chặn O')

# O có 4 quân dọc VÀ X có 4 quân ngang → O phải chọn thắng ngay
b = make_board(
    [(8, 2), (8, 3), (8, 4), (8, 5)],
    [(0, 3), (1, 3), (2, 3), (3, 3)],
)
o_wins = find_winning_actions(b, PLAYER_O)
x_threats = find_winning_actions(b, PLAYER_X)
print(f'Nước thắng ngay của O: {o_wins}')
print(f'Đe dọa của X: {x_threats}')
test_scenario(b, PLAYER_O, 'win_now', o_wins, 'Xung đột: O thắng > chặn X')

In [ ]:
print('=== PHẦN 4E: Trường hợp biên — bàn trống và bàn gần đầy ===')
print()

# Bàn cờ trống
b_empty = np.zeros((11, 11), dtype=np.int8)
test_scenario(b_empty, PLAYER_X, 'positional_fallback', [(5, 5)], 'Bàn trống — X đi nước đầu')

# Bàn cờ chỉ còn 1 ô trống
b_one = np.ones((11, 11), dtype=np.int8)
b_one[10, 10] = EMPTY
print(f'Ô trống duy nhất: (10, 10)')
_agent_one = RuleBasedAgent()
action_one = _agent_one.choose_action(b_one, PLAYER_X)
print(f'Nước chọn: {action_one} (kỳ vọng: (10, 10))')
print(f'Kết quả: {"✓ PASS" if action_one == (10, 10) else "✗ FAIL"}')

In [ ]:
print('=== PHẦN 4F: Thống kê nodes_visited và thinking_time ===')
print()

_agent_stats = RuleBasedAgent()
b_test = make_board([(5, 1), (5, 2), (5, 3), (5, 4)], [(0, 0), (0, 2), (0, 4)])

action = _agent_stats.choose_action(b_test, PLAYER_X)
print(f'Nước chọn: {action}')
print(f'nodes_visited: {_agent_stats.stats["nodes_visited"]}')
print(f'thinking_time: {_agent_stats.stats["thinking_time"]:.6f} giây')
print(f'Thông tin quyết định: {_agent_stats.explain_last_action()}')

---

## Phần 5 — Định nghĩa pattern và độ ưu tiên

> **Trạng thái:** Chưa thực hiện — sẽ hoàn thiện ở Giai đoạn 2.

Nội dung dự kiến:
- Định nghĩa các pattern: open-four, open-three, chuỗi có khoảng trống, chuỗi bị chặn một đầu.
- Bảng điểm trọng số và thứ tự ưu tiên khi nhiều pattern cùng kích hoạt.
- Ví dụ đối chứng: nhận diện đúng vs. nhận diện nhầm (false positive).


## Phần 6 — Giải thích quyết định nước đi

> **Trạng thái:** Chưa thực hiện — sẽ hoàn thiện ở Giai đoạn 2.

Nội dung dự kiến:
- Trực quan hóa luật được kích hoạt tại từng vị trí cờ.
- Phân tích các tình huống xung đột luật phức tạp.
- Dùng `explain_last_action()` để trace lý do chọn nước.


## Phần 7 — Đánh giá tác dụng thành phần (Ablation)

> **Trạng thái:** Chưa thực hiện — sẽ hoàn thiện ở Giai đoạn 3.

Nội dung dự kiến:
- So sánh Full Pattern (`use_patterns=True`) vs. No Pattern (`use_patterns=False`).
- Giữ nguyên luật thắng/chặn ở cả hai biến thể.
- Tập vị trí kiểm tra, seed và cơ chế phá hòa đồng nhất.


## Phần 8 — Thi đấu thử nghiệm

> **Trạng thái:** Chưa thực hiện — sẽ hoàn thiện ở Giai đoạn 3.

Nội dung dự kiến:
- Đấu với `RandomAgent`, đổi bên X/O, nhiều seed.
- Ghi nhận W/D/L, điểm số và `elapsed_s` trung bình.
- Dùng `evaluation/metrics.py` để tổng hợp kết quả.


## Phần 9 — Phân tích các trường hợp thất bại

> **Trạng thái:** Chưa thực hiện — sẽ hoàn thiện ở Giai đoạn 3.

Nội dung dự kiến:
- Phân tích thế cờ đối phương tạo đe dọa kép (double threat).
- Nhận diện nhầm pattern hoặc giới hạn của tầng luật hiện tại.
- Ghi nhận như hạn chế kỹ thuật, không yêu cầu giải quyết bài toán bất khả thi.


## Phần 10 — Tổng kết và hướng dẫn tái lập

> **Trạng thái:** Phần tổng kết Giai đoạn 1 bên dưới; phần số liệu đầy đủ sẽ cập nhật ở Giai đoạn 3.

### Tổng kết Giai đoạn 1

**Câu trả lời tạm thời (chưa có số liệu thi đấu chính thức):**  
Luật thắng ngay và chặn thua hoạt động chính xác ở cả 4 hướng và 2 phía (X/O) theo kết quả kiểm thử tự động. Nước đi dự phòng vị trí ưu tiên tâm bàn cờ hợp lý cho giai đoạn đầu ván cờ.

**Hạn chế kỹ thuật đã biết:**
- Fallback chưa nhận diện các hình thái chiến thuật cơ bản (open-three, open-four).
- Đe dọa kép (double threat) không thể phòng thủ hoàn toàn trong 1 nước.
- Bán kính ứng viên fallback = 2 có thể bỏ sót vị trí chiến lược xa trung tâm quân cờ.

**Cấu hình đề xuất bàn giao:**  
`RuleBasedAgent(candidate_radius=2, use_patterns=False, seed=None)`

**Thứ tự chạy lại notebook:**  
Restart Kernel → Run All. Không phụ thuộc vào trạng thái biến từ cell trước.
